# Notebook 1 — Des moteurs à la pose du gripper *(version d'étude commentée)*

> **À quoi sert ce notebook.** Le code de ce projet a été écrit pour résoudre la chaîne **perception → planification → contrôle**. Cette série de notebooks le **décortique pas à pas, en français**, pour que tu comprennes — et puisses défendre devant le jury — chaque brique. Pour chaque étape :
> - un **rappel théorique** avec renvoi aux concepts de cours,
> - le **vrai code** du dépôt (`src/…`), commenté,
> - une **exécution sur tes vraies données**, avec interprétation,
> - les **limites** connues.
>
> **Comment l'utiliser.** Lis dans l'ordre, exécute les cellules (`Run All`), modifie les valeurs pour voir ce qui change. Ce notebook **importe** les modules du dépôt : il montre le code réel, il ne le duplique pas — donc ce que tu apprends ici est exactement ce qui tourne sur le robot.
>
> *Note sur les renvois de cours.* J'écris les liens sous forme de concept (« algèbre linéaire : changement de base »). Donne-moi les noms/numéros exacts de tes cours UNIGE (vision, robotique, algèbre) et je les citerai précisément, comme la version ETUDE cite *C2*, *C5*.

**Plan de la série.** **(1) cette fondation géométrique** · 2) hand-eye : caméras ↔ robot · 3) perception 3D · 4) planification & contrôle.

---
## La question de ce notebook

Quand le robot lit ses capteurs, il obtient **6 entiers** (un par moteur Feetech STS3215), chacun entre `0` et `4095`. Par exemple, une vraie capture de ton dataset donne :

```
shoulder_pan=2031  shoulder_lift=860  elbow_flex=3084  wrist_flex=2373  wrist_roll=1049  gripper=1807
```

La question fondatrice est :

> **À partir de ces 6 entiers bruts, où se trouve physiquement la pince dans le repère du robot ?**

Y répondre demande **deux briques**, dans cet ordre :

| # | Brique | Entrée → Sortie | Fichier |
|---|---|---|---|
| 1 | **Encodeur → angle** | entier brut `0–4095` → angle articulaire (rad) | `src/calibration/motor_to_angle.py` |
| 2 | **Angles → pose** | 5 angles → position+orientation de la pince | `src/calibration/forward_kinematics.py` |

Le résultat final, qu'on note $T^{\text{base}}_{\text{gripper}}$, est une **matrice 4×4** qui dit *où* est la pince **et** *comment elle est orientée*, dans le repère base du robot. **Tout le reste du projet** (localiser un objet vu par les caméras, planifier une saisie) repose sur cette matrice. C'est la fondation — d'où le notebook 1.

In [ ]:
# === Configuration de l'environnement ===
# (à exécuter en premier ; trouve la racine du dépôt et rend les modules src/ importables)
import sys, json
from pathlib import Path
import numpy as np


def find_repo_root(start=None):
    """Remonte les dossiers jusqu'à trouver la racine (contient configs/ et src/)."""
    p = Path(start or Path.cwd()).resolve()
    for d in [p, *p.parents]:
        if (d / "configs").is_dir() and (d / "src").is_dir():
            return d
    raise RuntimeError("Racine du dépôt introuvable depuis " + str(p))


REPO = find_repo_root()
sys.path.insert(0, str(REPO))
print("Dépôt :", REPO)

# Données qu'on réutilisera dans tout le notebook
from src.calibration.motor_to_angle import load_motor_calibration
calib = load_motor_calibration(REPO / "configs" / "calibration_follower.json")

captures = json.load(open(REPO / "configs" / "extrinsic_capture_cam_0.json"))["captures"]
raw = captures[0]["motor_positions_raw"]          # une vraie capture (dict {moteur: entier})
print("Capture #1 (positions brutes) :", {k: int(v) for k, v in raw.items()})

---
## A. Rappel théorique — repères et transformations rigides

*(Rappel — **algèbre linéaire & géométrie** : changement de base, matrices de rotation, coordonnées homogènes.)*

Tout le projet manipule des **repères** (*frames*) : le repère de la base du robot, celui de la pince, celui de chaque caméra, celui d'un objet… Passer d'un repère à un autre se fait par une **transformation rigide** (rotation + translation, sans déformation) :

- une **rotation** $R$ : matrice $3\times3$ **orthonormale** avec $\det R = +1$ (on dit $R \in SO(3)$) ;
- une **translation** $t \in \mathbb{R}^3$.

On les empile dans une seule **matrice homogène** $4\times4$ (on dit $T \in SE(3)$) :

$$ T = \begin{bmatrix} R & t \\ 0\ 0\ 0 & 1 \end{bmatrix} $$

L'astuce des coordonnées homogènes : un point $p=(x,y,z)$ devient $\tilde p = (x,y,z,1)$, et **un seul produit matriciel** $T\,\tilde p$ applique d'un coup la rotation **et** la translation.

**Convention de nommage du projet** (à retenir, on l'utilise partout) :

$$ T^{A}_{B} = \text{pose du repère } B \text{ exprimée dans } A = \text{transforme un point de } B \text{ vers } A. $$

Deux règles qui découlent directement de l'algèbre linéaire :
- **Composition** (enchaîner) : $T^{A}_{C} = T^{A}_{B}\, T^{B}_{C}$ — les repères « intermédiaires » s'annihilent comme dans une simplification.
- **Inverse** : $\big(T^{A}_{B}\big)^{-1} = T^{B}_{A}$, et grâce à la structure rigide on n'inverse jamais une 4×4 « bêtement » : $R^{-1}=R^{\top}$, donc $T^{-1} = \begin{bmatrix} R^{\top} & -R^{\top}t \\ 0 & 1\end{bmatrix}$.

Le repère **pivot** du projet est le **repère base** (`base_link`) : origine au centre de la plaque posée sur la table, $X$ devant le robot, $Y$ à gauche, $Z$ vers le haut (détails dans `docs/REPERE_BASE.md`). Pour que la perception et le robot « parlent la même langue », **tout** finit exprimé dans ce repère.

> **À retenir.** Une matrice $4\times4$ de $SE(3)$ encode à la fois *où* et *comment orienté*. **Multiplier** des 4×4 = **enchaîner** des changements de repère. C'est l'unique outil géométrique de toute la chaîne — calibration, FK, triangulation, planification.

In [ ]:
# Les helpers SE(3) du projet (src/utils/transforms.py) implémentent exactement la théorie ci-dessus.
from src.utils.transforms import xyz_rpy_to_matrix, se3_inverse, se3_compose

# Exemple : un repère translaté de (0.1, 0.2, 0.3) m et tourné de 90° autour de Z
T = xyz_rpy_to_matrix([0.1, 0.2, 0.3], [0.0, 0.0, np.pi / 2])
print("T =\n", np.round(T, 3))

# Vérif 1 : R est bien une rotation (orthonormale, det = +1)
R = T[:3, :3]
print("\nR orthonormale (R Rᵀ = I) ?", np.allclose(R @ R.T, np.eye(3)))
print("det(R) = +1 ?            ", np.isclose(np.linalg.det(R), 1.0))

# Vérif 2 : T · T⁻¹ = identité  (l'inverse rigide ramène au point de départ)
print("\nT · se3_inverse(T) = I ?", np.allclose(se3_compose(T, se3_inverse(T)), np.eye(4)))

---
## B. Des encodeurs aux angles

### Pourquoi est-ce moins trivial qu'il n'y paraît ?

Chaque moteur Feetech STS3215 a un **encodeur 12 bits** : il mesure sa position sur **un tour** par un entier $\in [0, 4095]$ (donc $4096$ crans pour $360°$, soit $\approx 0{,}088°$ par cran). LeRobot convertit en degrés avec :

$$ \text{angle} = (\text{raw} - \text{mid}) \times \frac{360}{4095}, \qquad \text{mid} = \frac{\text{range\_min} + \text{range\_max}}{2}. $$

`mid` est le **milieu de la course calibrée** : on le pose comme « angle 0 ». Simple… **sauf que l'encodeur est circulaire** : après `4095`, il **resaute à `0`** (comme une boussole qui repasse de 359° à 0°). Si la course d'un moteur **chevauche cette couture `0/4095`**, deux positions physiquement voisines donnent des `raw` aux deux extrêmes (`4090` et `5`), et la formule naïve les croit séparées de presque 360°. C'est précisément le bug qui a bloqué `wrist_roll` sur ce robot.

### Parenthèse de compréhension — *l'encodeur circulaire* (

Prenons un **balayage physique continu** de `wrist_roll` qui passe sur la couture. Le poignet tourne doucement ; chaque lecture diffère de la précédente de $\sim 30$–$60$ crans :

| ordre | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|
| `raw` | 3980 | 4040 | 4090 | **20** | 70 | 130 |

Entre la lecture **3** (`4090`) et la **4** (`20`), le poignet n'a bougé que de $\sim$26 crans ($\approx 2°$). Mais en valeur brute ça **« saute »** de $-4070$ : on a traversé la couture.

**1. La formule naïve se trompe.** Avec $\text{mid} = \frac{20+4090}{2} = 2055$ :

$$ \text{angle}(4090) = (4090-2055)\tfrac{360}{4095} \approx +178{,}9°, \qquad \text{angle}(20) = (20-2055)\tfrac{360}{4095} \approx -178{,}9°. $$

→ amplitude apparente de **$\approx 358°$** pour un mouvement réel de $\approx 22°$. La pose calculée serait fausse de presque un demi-tour.

**2. La correction : « dérouler » autour d'un centre dans la bande utile.** On définit une fonction qui ramène toute différence d'encodeur dans l'intervalle $(-2048, +2048]$ (donc $(-180°, +180°]$) :

$$ \text{wrap}(d) = \big((d + 2048) \bmod 4096\big) - 2048. $$

Puis $\text{angle} = \text{wrap}(\text{raw} - \text{center}) \times \frac{360}{4095}$, où `center` est choisi **à l'intérieur** de la course réelle (ici $\approx 4040$). La soustraction modulo « recolle » les deux côtés de la couture : `4090` et `20` redeviennent voisins. On retrouve les $\approx 22°$ réels.

> **À retenir.** `wrap()` repose sur l'**arithmétique modulaire** (le `mod 4096`) : c'est la bonne façon de mesurer un écart sur un cercle. Pour un moteur dont la course **ne** touche **pas** la couture, `wrap()` ne change rien (no-op) — il n'agit que comme filet de sécurité.

### ) Fin de la parenthèse

In [ ]:
# Démonstration chiffrée : la formule naïve vs le déroulage circulaire du projet.
from src.calibration.motor_to_angle import wrap_encoder_delta

sweep = [3980, 4040, 4090, 20, 70, 130]   # un balayage physique continu traversant la couture
print("Balayage brut          :", sweep)
print("Écarts successifs       :", [sweep[i+1] - sweep[i] for i in range(len(sweep) - 1)])
print("  -> entre 4090 et 20 : physiquement +26 crans, mais le brut 'saute' de -4070\n")

# 1) Formule naïve : center = milieu (min, max)
mid = (min(sweep) + max(sweep)) / 2
naif = [(r - mid) * 360 / 4095 for r in sweep]
print(f"NAÏF  (center={mid}) : {[round(a, 1) for a in naif]}")
print(f"      amplitude apparente = {max(naif) - min(naif):.0f}°  (faux : mouvement réel ~22°)\n")

# 2) Déroulage circulaire : center DANS la bande utile
center = 4040
deroule = [wrap_encoder_delta(r - center) * 360 / 4095 for r in sweep]
print(f"DÉROULÉ (center={center}) : {[round(a, 1) for a in deroule]}")
print(f"      amplitude réelle = {max(deroule) - min(deroule):.1f}°  (correct et continu)")

### Ce qui s'est réellement passé sur ce robot

> Lors d'une session précédente, `wrist_roll` sortait des angles incohérents. Diagnostic : sa **course physique ($\sim 334°$) chevauchait la couture `0/4095`** ; sur 130 captures, ses lectures se répartissaient **près de `0` ET près de `4095`** — la signature exacte d'une couture traversée.
>
> **Correction (décision D1 du `PROJECT_STATUS.md`)** : on a *re-calé* le moteur (`scripts/measure_wrist_roll.py` puis `scripts/fix_wrist_roll_calibration.py`) pour réécrire son *Homing_Offset* afin que la couture tombe **hors** de la course utile (plage finale `[150, 3944]`). Depuis, la couture n'est plus jamais traversée en usage normal, et `motor_to_angle.py` reste **déroulage-aware** par sécurité.

La fonction de conversion réelle du projet est `raw_to_radians`. Appliquons-la à notre vraie capture.

In [ ]:
from src.calibration.motor_to_angle import raw_to_radians
from src.calibration.forward_kinematics import ARM_JOINTS   # les 5 articulations du bras, dans l'ordre

# ARM_JOINTS = chaîne base -> pince. Le 6e moteur (gripper) ouvre/ferme la pince :
# il ne déplace PAS la pince dans l'espace, donc il n'entre pas dans la cinématique.
print("Articulations de la chaîne :", ARM_JOINTS, "\n")

angles_rad = {j: raw_to_radians(raw[j], calib[j]) for j in ARM_JOINTS}
for j in ARM_JOINTS:
    c = calib[j]
    print(f"  {j:14s} raw={int(raw[j]):4d}  ->  {np.rad2deg(angles_rad[j]):+7.1f}°   "
          f"(plage calibrée [{c['range_min']}, {c['range_max']}])")

---
## C. La cinématique directe (Forward Kinematics, FK)

### Rappel théorique

*(Rappel — **robotique / géométrie** : chaînes cinématiques, composition de transformations rigides.)*

On a maintenant 5 **angles**. La **cinématique directe** répond à : *« connaissant les angles, où est la pince ? »*

Le bras est une **chaîne** de segments rigides reliés par des articulations rotoïdes :

$$ \texttt{base\_link} \xrightarrow{\text{shoulder\_pan}} \texttt{shoulder} \xrightarrow{\text{shoulder\_lift}} \texttt{upper\_arm} \xrightarrow{\text{elbow\_flex}} \texttt{lower\_arm} \xrightarrow{\text{wrist\_flex}} \texttt{wrist} \xrightarrow{\text{wrist\_roll}} \texttt{gripper} $$

Pour chaque articulation, le modèle géométrique (le fichier **URDF**) donne deux choses : une transformation **fixe** $T_{\text{origin}}$ (où est plantée l'articulation par rapport à la précédente, mesurée sur la CAO) et l'**axe** de rotation. La pose de la pince est le **produit le long de la chaîne** :

$$ T^{\text{base}}_{\text{gripper}} = \prod_{i} \; T_{\text{origin}}^{(i)} \cdot \mathrm{Rot}\big(\text{axe}_i,\ \theta_i\big). $$

C'est exactement la **règle de composition** $T^A_C = T^A_B T^B_C$ de la section A, appliquée 5 fois. Chaque facteur change de repère « base → segment 1 → segment 2 → … → pince ».

> **Choix d'ingénierie (décision D4).** Le modèle vient de l'URDF officiel `so101_new_calib.urdf` (CAO Onshape de TheRobotStudio), parsé en `numpy` pur. **L'URDF est l'unique source de vérité géométrique** : pour corriger le modèle on remplace le fichier, pas le code. On a écarté `pinocchio`/`placo` (le stack lourd de LeRobot), boîte noire difficile à défendre dans un mémoire.

In [ ]:
from src.calibration.forward_kinematics import KinematicChain

chain = KinematicChain()                       # parse configs/so101_new_calib.urdf une fois
print("Articulations actionnées :", chain.actuated)

# FK sur notre vraie capture
T_base_gripper = chain.fk(angles_rad)
pos_mm = T_base_gripper[:3, 3] * 1000.0
print("\nT_base_gripper =\n", np.round(T_base_gripper, 3))
print(f"\n=> Pince à (x={pos_mm[0]:.1f}, y={pos_mm[1]:.1f}, z={pos_mm[2]:.1f}) mm dans le repère base")

# Garde-fous : la rotation doit rester valide et l'échelle plausible (bras ~ tens of cm)
R = T_base_gripper[:3, :3]
print("\nRotation valide ?", np.allclose(R @ R.T, np.eye(3)) and np.isclose(np.linalg.det(R), 1.0))
print(f"Distance base→pince : {np.linalg.norm(pos_mm):.0f} mm (plausible si 50–600 mm)")

### Interprétation

Pour la capture #1, la FK place la pince à $(232, 18, 77)$ mm : à $\sim$23 cm devant la base, presque centrée en $Y$, assez basse ($Z=77$ mm) — cohérent avec un bras penché en avant vers la zone de travail.

**Test de cohérence parlant** : à la **configuration zéro** (tous les angles à 0, c.-à-d. au milieu de chaque course), la FK donne $(391, 0, 227)$ mm. C'est exactement la **pose « home »** documentée dans `docs/REPERE_BASE.md` ($\approx (391, 0, 227)$ mm) — preuve que notre FK et la convention de repère sont d'accord.

> **À retenir.** La FK transforme *5 angles* en *une pose 4×4* par 5 produits matriciels. C'est déterministe et exact (aux erreurs du modèle près, cf. limites). L'opération inverse — *« je veux la pince ICI, quels angles ? »* — c'est la **cinématique inverse (IK)**, utilisée plus tard pour commander le robot (notebook 4).

---
## D. Validation & limites

**Comment on sait que ça marche.**
- Chaque module a un **auto-test** (`python -m src.calibration.forward_kinematics`, idem `motor_to_angle`, `transforms`) qui vérifie : rotations valides, continuité à la couture, échelle plausible.
- Le script `scripts/check_calibration.py` rejoue tout d'un bloc et conclut `[OK]`.

**Limites à connaître (et à mentionner dans le mémoire).**
- **L'URDF est utilisé « tel quel »** : aucune calibration des décalages articulaires propres à *ton* exemplaire de robot. Les petits écarts de montage (jeu, impression 3D) ne sont pas modélisés ; ils se retrouvent absorbés plus tard dans les résidus de la calibration hand-eye (notebook 2).
- **Convention `new_calib` (décision D5)** : le « zéro » de chaque articulation = milieu de sa plage. C'est ce qui rend légitime le `mid = (range_min+range_max)/2` de la partie B — les deux conventions doivent rester cohérentes.
- **Répétabilité mécanique du SO-101** : de l'ordre de quelques mm. La FK est exacte mathématiquement, mais la pince réelle n'atteint pas l'angle commandé au degré près. C'est une limite *hardware*, pas *logicielle*.

---
## Récapitulatif & ce qui vient ensuite

**Ce que fait chaque brique vue ici :**

| Module | Rôle | Entrée → Sortie |
|---|---|---|
| `src/utils/transforms.py` | algèbre $SE(3)$ : composer/inverser des poses 4×4 | matrices ↔ matrices |
| `src/calibration/motor_to_angle.py` | encodeur circulaire → angle, **déroulage-aware** | entier `0–4095` → rad |
| `src/calibration/forward_kinematics.py` | cinématique directe depuis l'URDF | 5 angles → $T^{\text{base}}_{\text{gripper}}$ |

**Le fil rouge :** `6 entiers bruts` → (B) `angles` → (C) `T_base_gripper`. On sait maintenant **où est la pince**.

**Notebook 2 — Hand-eye.** Il reste à savoir **où sont les caméras** par rapport à la base. On filmera un damier tenu par la pince : pour chaque pose, on connaît $T^{\text{base}}_{\text{gripper}}$ (grâce à CE notebook) **et** la position du damier vue par la caméra. Résoudre le système donne $T^{\text{base}}_{\text{cam}}$ — et alors un objet vu par une caméra devient une position dans le repère base. C'est le pont entre *voir* et *agir*.

> **Pistes pour t'approprier ce notebook** (dans l'esprit de ta version de rendu Data Mining) : change la capture analysée (`captures[10]` au lieu de `captures[0]`), trace la position de la pince pour les 60 captures, ou vérifie « à la main » un produit de 2 matrices de la chaîne. Reformule les rappels théoriques avec tes mots — c'est cette réécriture qui ancre la compréhension.